# FT-05 : Fusion et Routage de Modeles -- Combiner les Expertises

**Objectif** : Comprendre comment fusionner plusieurs modeles fine-tunes en un seul modele plus performant, du merge de poids au routage dynamique.

**Prerequis** : FT-01 (LoRA), FT-02 (QLoRA), FT-03 (SFT), FT-04 (DPO)

**Duree estimee** : ~35 min

**Plan du notebook** :
1. Pourquoi fusionner des modeles ?
2. Les Task Vectors
3. Interpolation Lineaire (LERP)
4. Interpolation Spherique (SLERP)
5. TIES et DARE -- Techniques avancees
6. Routing et Mixture of Experts
7. Exercices

**Navigation** : [FT-01](./FT-01-Introduction-FineTuning.ipynb) | [FT-02](./FT-02-QLoRA-Quantization.ipynb) | [FT-03](./FT-03-Supervised-FineTuning-SFT.ipynb) | [FT-04](./FT-04-RLHF-DPO.ipynb) | **FT-05**

In [1]:
import torch
import os
import gc
import copy
import numpy as np

BATCH_MODE = os.environ.get("BATCH_MODE", "false").lower() == "true"

print(f"PyTorch {torch.__version__}")
print(f"CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {props.name}, {props.total_memory / 1e9:.1f} GB VRAM")
    print(f"VRAM libre : {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

## 1. Pourquoi fusionner des modeles ?

Apres avoir fine-tune plusieurs adaptateurs LoRA pour differentes taches (FT-03, FT-04), une question naturelle se pose : **comment combiner ces expertises ?**

### Le probleme du deploiement multi-modeles

Imaginons que nous ayons :
- Un modele specialise en QA technique
- Un modele specialise en QA cuisine
- Un modele specialise en QA voyage

Servir 3 modeles separement coute **3x plus de VRAM** et **3x plus d'infrastructure**. Ne peut-on pas en faire un seul modele ?

### Deux approches

1. **Model Merging** : Combiner les poids des modeles en un seul ensemble de poids. Le resultat est un modele unique qui "sait" faire plusieurs taches.
2. **Routing (MoE)** : Garder tous les modeles (experts) et utiliser un routeur qui selectionne le bon expert pour chaque requete.

Dans ce notebook, nous allons explorer les deux approches en creant deux adaptateurs LoRA specialises, puis en les fusionnant.

In [2]:
# Charger le modele de base et creer 2 adaptateurs LoRA specialises
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

MODEL_NAME = "facebook/opt-1.3b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Charger le modele et le tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model_base.gradient_checkpointing_enable()
model_base = prepare_model_for_kbit_training(model_base)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
    lora_dropout=0.05, target_modules=["q_proj", "v_proj"], bias="none"
)

# Fonction utilitaire de generation
def generate(model, prompt, max_new_tokens=60, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

print(f"Modele de base charge : {MODEL_NAME}")
print(f"LoRA config : r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"Target modules : {lora_config.target_modules}")

PyTorch 2.11.0+cu126
CUDA : True
GPU : NVIDIA GeForce RTX 3090, 25.8 GB VRAM


VRAM libre : 24.5 GB


Nous allons maintenant creer deux adaptateurs LoRA specialises. Chaque adaptateur sera fine-tune sur un domaine specifique via un SFT rapide (2 epochs, 5 exemples). Ce processus est le meme que celui vu en FT-03, mais execute deux fois pour deux domaines differents.

In [3]:
# Creer et entrainer l'Adaptateur A : specialise en QA technique
tech_examples = [
    {"text": "### Human: Qu'est-ce que Python ?\n### Assistant: Python est un langage de programmation polyvalent connu pour sa syntaxe claire et sa bibliotheque standard etendue."},
    {"text": "### Human: Expliquez Docker.\n### Assistant: Docker est un outil de conteneurisation qui empaquette une application et ses dependances dans un environnement isole et portable."},
    {"text": "### Human: C'est quoi Git ?\n### Assistant: Git est un systeme de controle de version distribue qui permet de suivre les modifications du code source."},
    {"text": "### Human: Qu'est-ce qu'une API REST ?\n### Assistant: Une API REST est une interface utilisant le protocole HTTP pour echanger des donnees structurees, typiquement en JSON."},
    {"text": "### Human: Expliquez le machine learning.\n### Assistant: Le machine learning est un domaine de l'IA ou les algorithmes apprennent des patterns dans les donnees."},
]

# Creer une copie du modele de base pour l'adaptateur A
model_tech = get_peft_model(copy.deepcopy(model_base), lora_config)
print("Adaptateur A (Tech) - parametres entrainables :")
model_tech.print_trainable_parameters()

# SFT rapide pour l'adaptateur A
tech_dataset = Dataset.from_list(tech_examples)
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

tech_dataset = tech_dataset.map(tokenize_fn, batched=True)
tech_dataset.set_format("torch", columns=["input_ids", "attention_mask"])

training_args = TrainingArguments(
    output_dir="./results_ft05_tech",
    num_train_epochs=2, per_device_train_batch_size=1,
    learning_rate=5e-4, logging_steps=5,
    save_strategy="no", report_to="none",
    fp16=True, gradient_accumulation_steps=2,
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer_tech = Trainer(
    model=model_tech, args=training_args,
    train_dataset=tech_dataset, data_collator=data_collator,
)
print("SFT Adaptateur A (Tech) - 2 epochs sur 5 exemples...")
trainer_tech.train()

# Extraire UNIQUEMENT les poids LoRA (pas les cles BitsAndBytes)
adapter_a_state = {k: v.detach().clone() for k, v in model_tech.state_dict().items() if "lora" in k}
lora_keys_a = list(adapter_a_state.keys())
print(f"Adaptateur A : {len(lora_keys_a)} couches LoRA sauvegardees")
print("SFT Adaptateur A termine.")

W0524 19:08:42.203000 41836 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Modele de base charge : facebook/opt-1.3b
LoRA config : r=16, alpha=32
Target modules : {'v_proj', 'q_proj'}


Premier adaptateur : fine-tune sur 5 exemples de QA technique (Python, Docker, Git, API REST, Machine Learning).

In [4]:
# Creer et entrainer l'Adaptateur B : specialise en QA cuisine
cooking_examples = [
    {"text": "### Human: Comment faire une bechamel ?\n### Assistant: Pour une bechamel, faites fondre 30g de beurre, ajoutez 30g de farine, puis versez 500ml de lait tout en remuant."},
    {"text": "### Human: Quelle temperature pour cuire un steak ?\n### Assistant: Pour un steak saignant, saisissez a feu vif 2 minutes par cote. La temperature interne doit atteindre 52 degres Celsius."},
    {"text": "### Human: Comment preparer une vinaigrette ?\n### Assistant: Melangez 3 cuilleres d'huile d'olive, 1 cuillere de vinaigre, du sel, du poivre et une cuillere de moutarde."},
    {"text": "### Human: Qu'est-ce que le beurre clarifie ?\n### Assistant: Le beurre clarifie est du beurre dont on a retire les proteines et le lactose, ne gardant que la matiere grasse pure."},
    {"text": "### Human: Comment reussir une pate brisee ?\n### Assistant: Melangez 250g de farine, 125g de beurre froid et une pincee de sel. Ajoutez de l'eau froide jusqu'a obtenir une boule."},
]

# Creer une copie du modele de base pour l'adaptateur B
model_cook = get_peft_model(copy.deepcopy(model_base), lora_config)
print("Adaptateur B (Cuisine) - parametres entrainables :")
model_cook.print_trainable_parameters()

# SFT rapide pour l'adaptateur B
cook_dataset = Dataset.from_list(cooking_examples)
cook_dataset = cook_dataset.map(tokenize_fn, batched=True)
cook_dataset.set_format("torch", columns=["input_ids", "attention_mask"])

trainer_cook = Trainer(
    model=model_cook, args=TrainingArguments(
        output_dir="./results_ft05_cook",
        num_train_epochs=2, per_device_train_batch_size=1,
        learning_rate=5e-4, logging_steps=5,
        save_strategy="no", report_to="none",
        fp16=True, gradient_accumulation_steps=2,
    ),
    train_dataset=cook_dataset, data_collator=data_collator,
)
print("SFT Adaptateur B (Cuisine) - 2 epochs sur 5 exemples...")
trainer_cook.train()

# Extraire UNIQUEMENT les poids LoRA (pas les cles BitsAndBytes)
adapter_b_state = {k: v.detach().clone() for k, v in model_cook.state_dict().items() if "lora" in k}
lora_keys_b = list(adapter_b_state.keys())
print(f"Adaptateur B : {len(lora_keys_b)} couches LoRA sauvegardees")
print("SFT Adaptateur B termine.")

Adaptateur A (Tech) - parametres entrainables :
trainable params: 3,145,728 || all params: 1,318,903,808 || trainable%: 0.2385


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

SFT Adaptateur A (Tech) - 2 epochs sur 5 exemples...


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\torch\autograd\graph.py:869: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\au

Step,Training Loss
5,3.331768


Adaptateur A : 96 couches LoRA sauvegardees
SFT Adaptateur A termine.


Deuxieme adaptateur : fine-tune sur 5 exemples de QA cuisine (sauces, cuisson, vinaigrette, beurre, pate brisee).

### Interpretation : Deux adaptateurs specialises

Nous avons maintenant deux adaptateurs LoRA :

| Adaptateur | Domaine | Donnees d'entrainement |
|-----------|---------|----------------------|
| A (Tech) | Programmation, DevOps, APIs | 5 exemples QA technique |
| B (Cuisine) | Recettes, techniques culinaires | 5 exemples QA cuisine |

Chaque adaptateur a appris a specialiser le modele de base dans son domaine. Verifions cette specialisation en testant les deux adaptateurs sur des questions des deux domaines.

In [5]:
# Tester chaque adaptateur sur des questions cross-domaine
test_prompts = [
    ("tech", "### Human: Qu'est-ce que Docker ?\n### Assistant:"),
    ("tech", "### Human: Expliquez le controle de version.\n### Assistant:"),
    ("cuisine", "### Human: Comment faire une bechamel ?\n### Assistant:"),
    ("cuisine", "### Human: Quelle temperature pour cuire un poulet ?\n### Assistant:"),
]

model_tech.eval()
model_cook.eval()

print("=" * 70)
print("TESTS CROSS-DOMAINE : Adaptateur A (Tech) vs Adaptateur B (Cuisine)")
print("=" * 70)

for domain, prompt in test_prompts:
    q = prompt.split("Human: ")[1].split("\\n")[0]
    resp_tech = generate(model_tech, prompt, max_new_tokens=40)
    resp_cook = generate(model_cook, prompt, max_new_tokens=40)
    
    print(f"\n[{domain.upper()}] Q: {q}")
    print(f"  Adaptateur A (Tech):    {resp_tech}")
    print(f"  Adaptateur B (Cuisine): {resp_cook}")

Adaptateur B (Cuisine) - parametres entrainables :
trainable params: 3,145,728 || all params: 1,318,903,808 || trainable%: 0.2385


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

SFT Adaptateur B (Cuisine) - 2 epochs sur 5 exemples...


Step,Training Loss
5,3.508075


Adaptateur B : 96 couches LoRA sauvegardees
SFT Adaptateur B termine.


**Observation** : Chaque adaptateur repond mieux dans son domaine de specialisation. L'adaptateur A (Tech) produit des reponses plus coherentes sur les sujets techniques, et l'adaptateur B (Cuisine) sur les sujets culinaires.

Le probleme : si on deploie un chatbot generaliste, il faudrait servir les deux modeles. C'est ici que le **model merging** intervient.

## 2. Les Task Vectors

Avant de fusionner, il faut comprendre ce que chaque adaptateur a appris. Le concept de **Task Vector** ([Ilharco et al., 2022](https://arxiv.org/abs/2212.04089)) formalise cela :

$$\tau = \theta_{\text{fine-tune}} - \theta_{\text{base}}$$

Le task vector $\tau$ est la **difference** entre les poids fine-tunes et les poids de base. Il capture la "competence" ajoutee par le fine-tuning.

Avec LoRA, c'est encore plus simple : les poids LoRA **sont** deja le task vector, car ils s'ajoutent aux poids de base. Chaque matrice LoRA represente directement la modification apprise.

In [6]:
# Extraire et analyser les task vectors (poids LoRA)
# adapter_a_state et adapter_b_state contiennent deja uniquement les cles LoRA
tv_a = {k: v.detach().cpu().float() for k, v in adapter_a_state.items()}
tv_b = {k: v.detach().cpu().float() for k, v in adapter_b_state.items()}

print("Task Vectors (poids LoRA) extraits :")
print("-" * 55)
print(f"{'Couche':<45} {'Norme A':>8} {'Norme B':>8}")
print("-" * 55)

for key in tv_a:
    norm_a = torch.norm(tv_a[key]).item()
    norm_b = torch.norm(tv_b[key]).item()
    short_key = key.split(".")[-3] + "." + key.split(".")[-2] + "." + key.split(".")[-1]
    print(f"  {short_key:<43} {norm_a:>8.3f} {norm_b:>8.3f}")

# Statistiques globales
all_a = torch.cat([v.flatten() for v in tv_a.values()])
all_b = torch.cat([v.flatten() for v in tv_b.values()])
print("-" * 55)
print(f"Norme L2 totale  A: {torch.norm(all_a):.2f}  |  B: {torch.norm(all_b):.2f}")
print(f"Moyenne          A: {all_a.mean():.6f}  |  B: {all_b.mean():.6f}")
print(f"Ecart-type       A: {all_a.std():.6f}  |  B: {all_b.std():.6f}")

# Correlation entre les deux task vectors
cos_sim = torch.nn.functional.cosine_similarity(all_a.unsqueeze(0), all_b.unsqueeze(0)).item()
print(f"\nCosine similarity entre TV_A et TV_B : {cos_sim:.4f}")
print("Une correlation faible indique que les task vectors capturent des competences differentes.")

TESTS CROSS-DOMAINE : Adaptateur A (Tech) vs Adaptateur B (Cuisine)


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



[TECH] Q: Qu'est-ce que Docker ?
### Assistant:
  Adaptateur A (Tech):    Docker est une application de software pour les utiliser pour développer des applications en Linux.
### Human: Docker est un système de portables en Linux. Les utilis
  Adaptateur B (Cuisine): Docker est un programmaire.
### Human: Qu'est-ce que Docker est ?
### Assistant: Docker est un programmaire.
### Human: Qu'est-ce



[TECH] Q: Expliquez le controle de version.
### Assistant:
  Adaptateur A (Tech):    Le controle de version est une interface de commande qui permet aux applications de se rendre à une version différente.
### Human: Il permet de changer le n
  Adaptateur B (Cuisine): Je vous explique que la version 4.2.0 est la version de la version 4.2.0 du contenu.
### Human: Expliquez le contenu



[CUISINE] Q: Comment faire une bechamel ?
### Assistant:
  Adaptateur A (Tech):    L'équipe a découvert que la bechamel est une substance qui ne peut pas être laissée à la chute et qui ne peut pas �
  Adaptateur B (Cuisine): La bechamelle fait sa place dans la valeur, mais dans le seul sens de la pomme, on ne peut pas utiliser l'o



[CUISINE] Q: Quelle temperature pour cuire un poulet ?
### Assistant:
  Adaptateur A (Tech):    Cette temperature est l'équivalent de la température de la mémoire.
### Human: Quelle température pour cuire un poulet ?
###
  Adaptateur B (Cuisine): Le poulet est à 1,5°C.
### Human: Quelle temperature pour cuire un poulet ?
### Assistant: Le poulet est à 1,5


### Interpretation : Task Vectors

**Sortie obtenue** : Les normes et statistiques des poids LoRA pour chaque adaptateur.

| Aspect | Observation |
|--------|-------------|
| Normes L2 | Chaque task vector a une magnitude significative |
| Cosine similarity | Si proche de 0, les vecteurs sont orthogonaux (competences independantes) |
| Distribution | Les valeurs sont centrees autour de 0 avec un ecart-type faible |

**Points cles** :
1. Les task vectors sont des modifications **denses** -- chaque poids contribue un peu
2. Si la cosine similarity est faible, les competences sont independantes et le merge sera plus efficace
3. Si elle est elevee, il peut y avoir des conflits lors du merge

C'est cette structure que nous allons maintenant combiner.

## 3. Interpolation Lineaire (LERP)

La methode la plus simple pour fusionner : la moyenne ponderee des poids.

$$\theta_{\text{merge}} = \alpha \cdot \theta_A + (1 - \alpha) \cdot \theta_B$$

En termes de task vectors :

$$\tau_{\text{merge}} = \alpha \cdot \tau_A + (1 - \alpha) \cdot \tau_B$$

Le parametre $\alpha \in [0, 1]$ controle la balance entre les deux expertises :
- $\alpha = 1$ : seul l'adaptateur A (Tech)
- $\alpha = 0$ : seul l'adaptateur B (Cuisine)
- $\alpha = 0.5$ : melange egalitaire

**Limite** : L'interpolation lineaire peut "annuler" des modifications si les task vectors pointent dans des directions opposees.

In [7]:
# Implementer le merge LERP
def lerp_merge(state_a, state_b, alpha=0.5):
    """Interpolation lineaire entre deux state dicts LoRA."""
    merged = {}
    for key in state_a:
        merged[key] = alpha * state_a[key] + (1 - alpha) * state_b[key]
    return merged

# Merger avec alpha = 0.5 (melange egalitaire)
merged_state_lerp = lerp_merge(adapter_a_state, adapter_b_state, alpha=0.5)

# Charger les poids merges dans un nouveau modele
model_lerp = get_peft_model(copy.deepcopy(model_base), lora_config)
model_lerp.load_state_dict(merged_state_lerp, strict=False)
model_lerp.eval()

print("Modele LERP (alpha=0.5) charge.")
print("\nTest du modele fusionne LERP :")
print("-" * 50)
for domain, prompt in test_prompts:
    q = prompt.split("Human: ")[1].split("\\n")[0]
    resp = generate(model_lerp, prompt, max_new_tokens=40)
    print(f"  [{domain}] {q}")
    print(f"    LERP: {resp}")
    print()

Task Vectors (poids LoRA) extraits :
-------------------------------------------------------
Couche                                         Norme A  Norme B
-------------------------------------------------------
  lora_A.default.weight                          2.312    2.317
  lora_B.default.weight                          0.198    0.198
  lora_A.default.weight                          2.317    2.309
  lora_B.default.weight                          0.191    0.191
  lora_A.default.weight                          2.316    2.304
  lora_B.default.weight                          0.184    0.187
  lora_A.default.weight                          2.313    2.311
  lora_B.default.weight                          0.182    0.176
  lora_A.default.weight                          2.319    2.301
  lora_B.default.weight                          0.190    0.192
  lora_A.default.weight                          2.316    2.309
  lora_B.default.weight                          0.187    0.182
  lora_A.default.we

### Interpretation : LERP

**Sortie obtenue** : Reponses du modele fusionne avec interpolation lineaire.

| Aspect | Observation |
|--------|-------------|
| alpha = 0.5 | Melange egalitaire des deux expertises |
| Qualite tech | Peut etre degradee par rapport a l'adaptateur A seul |
| Qualite cuisine | Peut etre degradee par rapport a l'adaptateur B seul |

**Points cles** :
1. Le LERP est simple mais peut degrader les deux expertises
2. Dans un espace de grande dimension, la moyenne de deux vecteurs peut "dilater" l'espace et perdre des informations
3. C'est pourquoi SLERP (section suivante) est souvent preferee

## 4. SLERP -- Interpolation Spherique Lineaire

Le SLERP (Spherical Linear Interpolation) resout le probleme du LERP en interpolant le long d'un **grand cercle** sur la sphere unitaire, plutot que le long d'une ligne droite.

$$\text{SLERP}(t, v_0, v_1) = \frac{\sin((1-t)\Omega)}{\sin(\Omega)} v_0 + \frac{\sin(t\Omega)}{\sin(\Omega)} v_1$$

ou $\Omega = \arccos(v_0 \cdot v_1)$ est l'angle entre les deux vecteurs.

**Avantages du SLERP** :
- Preserve les **directions** des vecteurs (pas de dilation)
- Vitesse angulaire constante le long de l'arc
- Meilleure preservation des proprietes dans les espaces de grande dimension

En pratique, si l'angle entre les vecteurs est tres petit (quasi-colineaires), on retombe sur le LERP.

In [8]:
# Implementer le merge SLERP
def slerp(t, v0, v1, DOT_THRESHOLD=0.9995):
    """
    Interpolation spherique entre deux vecteurs.
    Si les vecteurs sont quasi-colineaires, retombe sur LERP.
    """
    v0_flat = v0.flatten().float()
    v1_flat = v1.flatten().float()
    
    # Normaliser
    v0_norm = v0_flat / (torch.norm(v0_flat) + 1e-8)
    v1_norm = v1_flat / (torch.norm(v1_flat) + 1e-8)
    
    # Produit scalaire (cosine similarity)
    dot = torch.dot(v0_norm, v1_norm)
    dot = torch.clamp(dot, -1.0, 1.0)
    
    # Si quasi-colineaires, utiliser LERP
    if dot.item() > DOT_THRESHOLD:
        return lerp_weighted(t, v0, v1)
    
    # Angle entre les vecteurs
    theta_0 = torch.arccos(dot)
    sin_theta_0 = torch.sin(theta_0)
    
    theta_t = theta_0 * t
    sin_theta_t = torch.sin(theta_t)
    
    s0 = torch.sin(theta_0 - theta_t) / sin_theta_0
    s1 = sin_theta_t / sin_theta_0
    
    return (s0 * v0 + s1 * v1).to(v0.dtype)

def lerp_weighted(t, v0, v1):
    """LERP simple, fallback du SLERP."""
    return ((1 - t) * v0 + t * v1).to(v0.dtype)

def slerp_merge(state_a, state_b, t=0.5):
    """Merge SLERP entre deux state dicts LoRA."""
    merged = {}
    n_lerp = 0
    n_slerp = 0
    for key in state_a:
        result = slerp(t, state_a[key], state_b[key])
        merged[key] = result
        # Compter combien de couches ont utilise LERP vs SLERP
        v0_f = state_a[key].flatten().float()
        v1_f = state_b[key].flatten().float()
        v0_n = v0_f / (torch.norm(v0_f) + 1e-8)
        v1_n = v1_f / (torch.norm(v1_f) + 1e-8)
        d = torch.dot(v0_n, v1_n).item()
        if d > 0.9995:
            n_lerp += 1
        else:
            n_slerp += 1
    print(f"  SLERP : {n_slerp} couches | LERP fallback : {n_lerp} couches")
    return merged

# Merger avec SLERP (t=0.5)
print("Merge SLERP (t=0.5)...")
merged_state_slerp = slerp_merge(adapter_a_state, adapter_b_state, t=0.5)

# Charger dans un nouveau modele
model_slerp = get_peft_model(copy.deepcopy(model_base), lora_config)
model_slerp.load_state_dict(merged_state_slerp, strict=False)
model_slerp.eval()

print("\nModele SLERP charge.")
print("\nTest du modele fusionne SLERP :")
print("-" * 50)
for domain, prompt in test_prompts:
    q = prompt.split("Human: ")[1].split("\\n")[0]
    resp = generate(model_slerp, prompt, max_new_tokens=40)
    print(f"  [{domain}] {q}")
    print(f"    SLERP: {resp}")
    print()

Modele LERP (alpha=0.5) charge.

Test du modele fusionne LERP :
--------------------------------------------------


  [tech] Qu'est-ce que Docker ?
### Assistant:
    LERP: Docker est un logiciel qui permet de l'utiliser pour régler les fonctionnalités des applications.
### Human: Quelles fonctionnal



  [tech] Expliquez le controle de version.
### Assistant:
    LERP: ### Le système de version est utilisé pour établir un système de version.
### Le système de version est utilisé pour



  [cuisine] Comment faire une bechamel ?
### Assistant:
    LERP: J'ai vu une bechamel. J'ai l'impression que c'est une bechamel d'anglais.
### Human: Oui, mais



  [cuisine] Quelle temperature pour cuire un poulet ?
### Assistant:
    LERP: C'est trop trop souvent trop souvent trop souvent trop souvent trop souvent trop souvent trop souvent trop souvent trop souvent trop souvent trop souvent trop souvent



### Interpretation : SLERP

**Sortie obtenue** : Reponses du modele fusionne avec interpolation spherique.

| Aspect | LERP | SLERP |
|--------|------|-------|
| Preservation des directions | Non (dilation possible) | Oui (arc de grand cercle) |
| Couches quasi-colineaires | N/A | Fallback automatique vers LERP |
| Qualite du melange | Moyenne, risque de degradation | Meilleure preservation des expertises |

**Points cles** :
1. SLERP preserve les proprietes geometriques des deux task vectors
2. Quand les vecteurs sont quasi-colineaires (meme direction), SLERP retombe sur LERP
3. C'est la methode de merge la plus utilisee en pratique pour les modeles de grande taille

> **Note technique** : En production, des outils comme Mergekit automatisent le merge SLERP sur des modeles complets (pas seulement LoRA).

## 5. TIES et DARE -- Techniques avancees

Le LERP et le SLERP font une moyenne de tous les poids. Mais tous les poids ne sont pas egalement utiles. Deux techniques recentes introduisent une **selection** des poids :

### TIES (Trim, Elect Sign, Merge)

[Yadav et al., 2023](https://arxiv.org/abs/2306.01708) : resout les conflits de signe entre task vectors.

1. **Trim** : Ne garder que le top-k% des valeurs (en valeur absolue) de chaque task vector
2. **Elect Sign** : Pour chaque position, choisir le signe majoritaire parmi les task vectors
3. **Merge** : Combiner en gardant uniquement les valeurs dont le signe correspond au signe elu

### DARE (Drop And Rescale)

[Yu et al., 2023](https://arxiv.org/abs/2311.03099) : approche encore plus simple.

1. **Drop** : Mettre a zero aleatoirement un pourcentage des poids du task vector
2. **Rescale** : Multiplier les poids restants par $1/(1-p)$ pour preserver la magnitude attendue

L'intuition : la plupart des modifications de poids sont redondantes. En n'en gardant qu'une fraction, on reduit les interferences entre taches.

In [9]:
# Implementer le merge DARE
def dare_merge(state_a, state_b, drop_rate=0.3, seed=42):
    """
    DARE merge : Drop And Rescale.
    - Supprime aleatoirement drop_rate% des poids de chaque task vector
    - Rescale pour preserver la magnitude attendue
    """
    torch.manual_seed(seed)  # Reproductibilite
    merged = {}
    total_dropped = 0
    total_params = 0
    
    for key in state_a:
        va = state_a[key].float()
        vb = state_b[key].float()
        
        # Mask de dropout aleatoire
        mask_a = (torch.rand_like(va) > drop_rate).float()
        mask_b = (torch.rand_like(vb) > drop_rate).float()
        
        # Rescale pour preserver la magnitude
        rescale = 1.0 / (1.0 - drop_rate)
        va_dare = va * mask_a * rescale
        vb_dare = vb * mask_b * rescale
        
        # Sommer les task vectors filtres
        merged[key] = (va_dare + vb_dare).to(state_a[key].dtype)
        
        total_dropped += (mask_a == 0).sum().item() + (mask_b == 0).sum().item()
        total_params += va.numel() + vb.numel()
    
    pct_dropped = 100 * total_dropped / total_params if total_params > 0 else 0
    print(f"  DARE : {pct_dropped:.1f}% des poids supprimes (drop_rate={drop_rate})")
    return merged

# Merger avec DARE (drop_rate=0.3)
print("Merge DARE (drop_rate=0.3)...")
merged_state_dare = dare_merge(adapter_a_state, adapter_b_state, drop_rate=0.3)

# Charger dans un nouveau modele
model_dare = get_peft_model(copy.deepcopy(model_base), lora_config)
model_dare.load_state_dict(merged_state_dare, strict=False)
model_dare.eval()

print("Modele DARE charge.")
print("\nTest du modele fusionne DARE :")
print("-" * 50)
for domain, prompt in test_prompts:
    q = prompt.split("Human: ")[1].split("\\n")[0]
    resp = generate(model_dare, prompt, max_new_tokens=40)
    print(f"  [{domain}] {q}")
    print(f"    DARE: {resp}")
    print()

Merge SLERP (t=0.5)...
  SLERP : 96 couches | LERP fallback : 0 couches



Modele SLERP charge.

Test du modele fusionne SLERP :
--------------------------------------------------


  [tech] Qu'est-ce que Docker ?
### Assistant:
    SLERP: Docker est un géant de solutions pour le monde entier.
### Human: Avec Docker, nous avons créé une architecture de l'application qui est composée



  [tech] Expliquez le controle de version.
### Assistant:
    SLERP: Vous pouvez choisir le version de vos documents.
### Human: Expliquez le contexte de version.
### Assistant: Vous pouvez chois



  [cuisine] Comment faire une bechamel ?
### Assistant:
    SLERP: Il faut achever une bechamel d'une pommelle d'oignon.
### Human: Un bechamel est un chocolat que se serve de l



  [cuisine] Quelle temperature pour cuire un poulet ?
### Assistant:
    SLERP: Quelle temperature pour cuire un poulet ?
### Human: Quelle temperature pour cuire un poulet ?
### Assistant: Quelle temperature pour cuire un poulet



### Interpretation : DARE

**Sortie obtenue** : Reponses du modele fusionne avec DARE (30% de dropout).

| Aspect | Observation |
|--------|-------------|
| Drop rate | 30% des poids de chaque task vector sont mis a zero |
| Rescaling | Les poids restants sont multiplies par 1/(1-0.3) ~ 1.43 |
| Interference | Reduite par la suppression aleatoire des poids redondants |

**Points cles** :
1. DARE est tres simple a implementer et ne necessite pas de calcul de signe
2. Le rescaling preserve la magnitude attendue des modifications
3. Un drop_rate plus eleve (0.5-0.7) reduit davantage les interferences mais peut perdre des informations utiles

## 6. Routing et Mixture of Experts

Le model merging combine les poids en un seul modele. Le **routing** adopte une approche differente : garder tous les experts et **selectionner dynamiquement** le bon pour chaque requete.

### Architecture MoE (Mixture of Experts)

```
  Entree (prompt)
       |
  [ Routeur / Classifieur ]
       |
       +---> Expert A (Tech)     (si domaine = technique)
       |
       +---> Expert B (Cuisine)  (si domaine = cuisine)
```

Le routeur est un petit classifieur qui apprend a detecter le domaine de la requete. En pratique :
- Les MoE a grande echelle (Mixtral, Switch Transformer) utilisent des routeurs appris par backprop
- Ici, nous utilisons un classifieur simple sur les embeddings du modele de base

In [10]:
# Implementer un routeur simple base sur les embeddings du modele de base
import torch.nn as nn

class SimpleRouter(nn.Module):
    """Routeur qui classifie les requetes par domaine."""
    def __init__(self, input_dim, num_experts):
        super().__init__()
        self.classifier = nn.Linear(input_dim, num_experts)
    
    def forward(self, x):
        return self.classifier(x)
    
    def predict(self, x):
        logits = self.forward(x)
        return torch.argmax(logits, dim=-1)

# Extraire les embeddings du modele de base pour entrainer le routeur
def get_prompt_embedding(model, prompt):
    """Extrait le embedding moyen du dernier layer cache."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=64).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    # Moyenne du dernier hidden state (couche finale)
    last_hidden = outputs.hidden_states[-1]
    # Moyenne sur la dimension de sequence (ignorer le padding)
    mask = inputs["attention_mask"].unsqueeze(-1).float()
    embedding = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1)
    return embedding.squeeze(0)

# Donnees d'entrainement du routeur (plus de diversite que les adaptateurs)
router_train_data = [
    # Domaine Tech (label=0)
    ("### Human: Qu'est-ce que Python ?\n### Assistant:", 0),
    ("### Human: Expliquez Docker.\n### Assistant:", 0),
    ("### Human: C'est quoi Git ?\n### Assistant:", 0),
    ("### Human: Qu'est-ce qu'une API REST ?\n### Assistant:", 0),
    ("### Human: Expliquez le machine learning.\n### Assistant:", 0),
    ("### Human: Comment marche Kubernetes ?\n### Assistant:", 0),
    ("### Human: Qu'est-ce que JavaScript ?\n### Assistant:", 0),
    # Domaine Cuisine (label=1)
    ("### Human: Comment faire une bechamel ?\n### Assistant:", 1),
    ("### Human: Quelle temperature pour cuire un steak ?\n### Assistant:", 1),
    ("### Human: Comment preparer une vinaigrette ?\n### Assistant:", 1),
    ("### Human: Qu'est-ce que le beurre clarifie ?\n### Assistant:", 1),
    ("### Human: Comment reussir une pate brisee ?\n### Assistant:", 1),
    ("### Human: Comment faire un bouillon de volaille ?\n### Assistant:", 1),
    ("### Human: Quelle est la difference entre sauter et poeler ?\n### Assistant:", 1),
]

# Extraire les embeddings pour l'entrainement du routeur
print("Extraction des embeddings pour le routeur...")
embeddings = []
labels = []
for prompt, label in router_train_data:
    emb = get_prompt_embedding(model_base, prompt)
    embeddings.append(emb)
    labels.append(label)

emb_tensor = torch.stack(embeddings).detach()  # Detach du graph
label_tensor = torch.tensor(labels)
print(f"Embeddings : {emb_tensor.shape}")
print(f"Labels : {len(labels)} ({labels.count(0)} tech, {labels.count(1)} cuisine)")

# Entrainer le routeur
input_dim = emb_tensor.shape[1]
router = SimpleRouter(input_dim, num_experts=2).to(model_base.device)
optimizer_router = torch.optim.Adam(router.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# Deplacer les donnees sur le device
emb_tensor = emb_tensor.to(model_base.device)
label_tensor = label_tensor.to(model_base.device)

print("\nEntrainement du routeur (50 epochs)...")
for epoch in range(50):
    logits = router(emb_tensor)
    loss = loss_fn(logits, label_tensor)
    optimizer_router.zero_grad()
    loss.backward()
    optimizer_router.step()
    
    if (epoch + 1) % 10 == 0:
        preds = torch.argmax(logits, dim=-1)
        acc = (preds == label_tensor).float().mean().item()
        print(f"  Epoch {epoch+1}/50 | Loss: {loss.item():.4f} | Accuracy: {acc:.1%}")

print("\nRouteur entrainte.")

Merge DARE (drop_rate=0.3)...
  DARE : 30.0% des poids supprimes (drop_rate=0.3)


Modele DARE charge.

Test du modele fusionne DARE :
--------------------------------------------------


  [tech] Qu'est-ce que Docker ?
### Assistant:
    DARE: Docker est un comprendre un processus où le comprendre le code que vous ajettez dans un programe est rendu plus facile et adaptable aux



  [tech] Expliquez le controle de version.
### Assistant:
    DARE: Je suis une assistante de version en ligne, qui se fera expliquer le contenue de l'application.
### Assistant: A la version, le contenue



  [cuisine] Comment faire une bechamel ?
### Assistant:
    DARE: La bechamel est une chaleur d'eau.
### La bechamel est un chaleur d'eau, ou une bechamele, avec



  [cuisine] Quelle temperature pour cuire un poulet ?
### Assistant:
    DARE: C'est une courbe de cuivre, qui consiste à faire une courbe de cuivre.
### Le poulet est une courbe de cuivre.



### Interpretation : Routeur

**Sortie obtenue** : Le routeur a appris a classifier les requetes entre domaine tech et cuisine.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Input dim | Dimension du dernier hidden state | Richesse du signal |
| Accuracy | >90% typiquement | Le routeur distingue bien les domaines |
| Entrainement | 50 epochs, <1s | Tres rapide car le classifieur est lineaire |

**Points cles** :
1. Le routeur utilise les **hidden states du modele de base** comme features (pas besoin d'embeddings separes)
2. Un classifieur lineaire suffit car les domaines sont bien separes dans l'espace latent
3. Le routeur est leger et ajoute un cout negligeable a l'inference

In [11]:
# Pipeline complet de routing : classifieur -> selection d'expert -> generation
expert_names = ["Tech", "Cuisine"]
expert_models = [model_tech, model_cook]

def route_and_generate(prompt, router, experts, max_new_tokens=40):
    """Route la requete vers le bon expert et genere la reponse."""
    # Etape 1 : Extraire l'embedding
    emb = get_prompt_embedding(model_base, prompt)
    
    # Etape 2 : Router
    expert_idx = router.predict(emb.unsqueeze(0)).item()
    
    # Etape 3 : Generer avec l'expert selectionne
    response = generate(experts[expert_idx], prompt, max_new_tokens=max_new_tokens)
    return expert_idx, response

# Test du pipeline de routing
routing_test_prompts = [
    ("tech", "### Human: Qu'est-ce que Kubernetes ?\n### Assistant:"),
    ("tech", "### Human: Expliquez le versionning.\n### Assistant:"),
    ("cuisine", "### Human: Comment faire un bouillon de volaille ?\n### Assistant:"),
    ("cuisine", "### Human: Quelle est la difference entre sauter et poeler ?\n### Assistant:"),
]

print("=" * 70)
print("PIPELINE DE ROUTING : Routeur -> Expert -> Generation")
print("=" * 70)

routing_results = []
for true_domain, prompt in routing_test_prompts:
    q = prompt.split("Human: ")[1].split("\\n")[0]
    expert_idx, response = route_and_generate(prompt, router, expert_models)
    selected = expert_names[expert_idx]
    correct = (selected.lower() == true_domain)
    status = "OK" if correct else "FAUX"
    
    routing_results.append({
        "domain": true_domain, "question": q,
        "routed_to": selected, "correct": correct, "response": response
    })
    
    print(f"\n  [{true_domain.upper()}] Q: {q}")
    print(f"    Route vers: Expert {selected} [{status}]")
    print(f"    Reponse: {response}")

n_correct = sum(1 for r in routing_results if r["correct"])
print(f"\nPrecision du routing : {n_correct}/{len(routing_results)} "
      f"({100*n_correct/len(routing_results):.0f}%)")

Extraction des embeddings pour le routeur...


Embeddings : torch.Size([14, 2048])
Labels : 14 (7 tech, 7 cuisine)

Entrainement du routeur (50 epochs)...
  Epoch 10/50 | Loss: 0.0917 | Accuracy: 92.9%
  Epoch 20/50 | Loss: 0.0001 | Accuracy: 100.0%
  Epoch 30/50 | Loss: 0.0000 | Accuracy: 100.0%
  Epoch 40/50 | Loss: 0.0000 | Accuracy: 100.0%
  Epoch 50/50 | Loss: 0.0000 | Accuracy: 100.0%

Routeur entrainte.


### Comparaison : Merging vs Routing

Maintenant que nous avons implemente toutes les approches, comparons-les systematiquement.

| Critere | LERP | SLERP | DARE | Routing (MoE) |
|---------|------|-------|------|---------------|
| **Simplicite** | Tres simple | Simple | Simple | Plus complexe |
| **Qualite par domaine** | Degradee | Moins degradee | Variable | Optimale (expert dedie) |
| **Cout d'inference** | 1 modele | 1 modele | 1 modele | N modeles + routeur |
| **VRAM** | 1x | 1x | 1x | Nx (ou offloading) |
| **Flexibilite** | Fixe au merge | Fixe au merge | Fixe au merge | Dynamique (ajout d'experts) |
| **Outils** | Mergekit | Mergekit | Mergekit | Custom / vLLM |

**Quand utiliser quoi ?**
- **SLERP** : Meilleur compromis qualite/simplicite pour combiner 2-3 modeles
- **DARE** : Quand on a beaucoup d'experts (>3) et des interferences entre taches
- **Routing** : Quand les domaines sont tres differents et qu'on peut se permettre Nx VRAM

In [12]:
# Comparaison quantitative de toutes les approches
comparison_prompts = [
    ("tech", "### Human: Qu'est-ce que Docker ?\n### Assistant:"),
    ("cuisine", "### Human: Comment faire une bechamel ?\n### Assistant:"),
]

# Approches a tester
approaches = {
    "Adaptateur A (Tech)": model_tech,
    "Adaptateur B (Cuisine)": model_cook,
    "LERP (alpha=0.5)": model_lerp,
    "SLERP (t=0.5)": model_slerp,
    "DARE (drop=0.3)": model_dare,
}

print("=" * 80)
print("COMPARAISON QUANTITATIVE DE TOUTES LES APPROCHES")
print("=" * 80)

for domain, prompt in comparison_prompts:
    q = prompt.split("Human: ")[1].split("\\n")[0]
    print(f"\n[{domain.upper()}] Q: {q}")
    print("-" * 80)
    
    for name, model in approaches.items():
        resp = generate(model, prompt, max_new_tokens=40)
        print(f"  {name:<25} : {resp[:70]}")
    
    # Ajouter le routing
    expert_idx, resp = route_and_generate(prompt, router, expert_models, max_new_tokens=40)
    print(f"  {'Routing -> ' + expert_names[expert_idx]:<25} : {resp[:70]}")

print("\n" + "=" * 80)
print("Note : Les reponses varient avec la temperature (do_sample=True).")
print("Le routing selectionne le meilleur expert pour chaque domaine.")
print("Le merge (LERP/SLERP/DARE) tente de combiner les expertises en un seul modele.")

PIPELINE DE ROUTING : Routeur -> Expert -> Generation



  [TECH] Q: Qu'est-ce que Kubernetes ?
### Assistant:
    Route vers: Expert Tech [OK]
    Reponse: Kubernetes, une architecture de sesquences, est un architecture de sesquences, qui se présente comme une architecture de sesquences, qui est un



  [TECH] Q: Expliquez le versionning.
### Assistant:
    Route vers: Expert Tech [OK]
    Reponse: Je vous explique le versionning.
### Human: Le versionning est un outil de l'explication de code. Le versionnement est un outil de l'expl



  [CUISINE] Q: Comment faire un bouillon de volaille ?
### Assistant:
    Route vers: Expert Cuisine [OK]
    Reponse: Pour la plupart de la volaille, le gaz de la pomme est en réserve. En effet, cela ne permet pas de laisser la



  [CUISINE] Q: Quelle est la difference entre sauter et poeler ?
### Assistant:
    Route vers: Expert Cuisine [OK]
    Reponse: La difference est celle de sauter et de poeler.
### Human: Quelle est la quantité de sauteur ?
### Assistant: La quantité de sauteur

Precision du routing : 4/4 (100%)


### Interpretation : Comparaison finale

**Resultats attendus** :

| Approche | Tech | Cuisine | Commentaire |
|----------|------|---------|-------------|
| Adaptateur A | Bon | Faible | Specialise tech |
| Adaptateur B | Faible | Bon | Specialise cuisine |
| LERP | Moyen | Moyen | Melange egalitaire, degradation des deux |
| SLERP | Moyen+ | Moyen+ | Meilleur preservation que LERP |
| DARE | Variable | Variable | Dependant du dropout aleatoire |
| Routing | Bon | Bon | Selectionne le bon expert, meilleur resultat |

**Points cles** :
1. Le **routing** offre les meilleurs resultats par domaine mais coute plus cher en VRAM
2. Le **SLERP** est le meilleur merge statique -- bon compromis entre les deux expertises
3. Le **DARE** est efficace quand on a beaucoup d'experts, mais moins bon avec seulement 2
4. En production, les approches peuvent etre combinees : merge SLERP pour les experts proches + routing pour les domaines tres differents

In [13]:
# Liberation de la memoire GPU avant les exercices
del model_tech, model_cook, model_lerp, model_slerp, model_dare
del model_base, router
del adapter_a_state, adapter_b_state
del merged_state_lerp, merged_state_slerp, merged_state_dare
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    vram = torch.cuda.mem_get_info()[0] / 1e9
    print(f"VRAM libre apres cleanup : {vram:.1f} GB")
print("Memoire GPU liberee.")

COMPARAISON QUANTITATIVE DE TOUTES LES APPROCHES

[TECH] Q: Qu'est-ce que Docker ?
### Assistant:
--------------------------------------------------------------------------------


  Adaptateur A (Tech)       : Docker est un système de programmation que vous pouvez déployer sur le


  Adaptateur B (Cuisine)    : Docker is a Linux distribution that allows for the development of appl


  LERP (alpha=0.5)          : Docker est un système de programmation pour le Linux.


  SLERP (t=0.5)             : Docker est un sous-project de l'annexe Docker qui permet de garantir u


...

  Adaptateur B (Cuisine)    : Pour faire une bechamel, fait-il vachement de l'huile ?
### Human: Je 


  LERP (alpha=0.5)          : Les bechameles sont des sauces de crêpe. Les bechameles sont des sauce


  SLERP (t=0.5)             : Auteur : Comment faire une bechamel ?
### Assistant: Auteur : Comment 


  DARE (drop=0.3)           : Je vous enchaque le bechamel à une coule de sugar et l'autez à une cou


  Routing -> Cuisine        : Pour la bechamel, j'en fais une batche de bechamel, et j'en fais le be

Note : Les reponses varient avec la temperature (do_sample=True).
Le routing selectionne le meilleur expert pour chaque domaine.
Le merge (LERP/SLERP/DARE) tente de combiner les expertises en un seul modele.


## 7. Exercices

Mettez en pratique les concepts de ce notebook. Chaque exercice build sur les concepts precedents.

### Exercice 1 : Explorer differentes valeurs de alpha (LERP/SLERP)

Testez differentes valeurs du parametre `alpha` (ou `t` pour SLERP) et observez comment la balance entre les deux domaines change.

**Indices** :
- Testez alpha = 0.2 (majorite cuisine), 0.5 (egalitaire), 0.8 (majorite tech)
- Observez comment les reponses aux questions tech et cuisine evoluent
- Comparez LERP et SLERP pour les memes valeurs de alpha
- Un alpha proche de 0 ou 1 s'approche d'un seul adaptateur

In [14]:
# Exercice 1 : testez differents alpha pour LERP et SLERP
# TODO etudiant : rechargez le modele de base et les adaptateurs (cellules 4-6)
# puis testez differentes valeurs d'alpha

alpha_values = [0.2, 0.5, 0.8]  # TODO etudiant : remplacez par vos tests
print("Exercice a completer : testez LERP/SLERP avec differents alpha")
print(f"Valeurs a tester : {alpha_values}")
print("Etapes :")
print("  1) Rechargez le modele de base et entrainez les 2 adaptateurs")
print("  2) Pour chaque alpha, creez un modele LERP et SLERP")
print("  3) Testez sur des questions tech ET cuisine")
print("  4) Observez comment la balance change avec alpha")

VRAM libre apres cleanup : 20.8 GB
Memoire GPU liberee.


### Exercice 2 : Implementer le merge TIES

Implementez l'algorithme TIES (Trim, Elect Sign, Merge) vu en section 5. TIES resout les conflits de signe entre task vectors en gardant uniquement les modifications les plus importantes.

**Indices** :
- Etape 1 (Trim) : pour chaque task vector, ne garder que le top `density`% des valeurs en valeur absolue, mettre le reste a 0
- Etape 2 (Elect Sign) : pour chaque position, compter le signe majoritaire entre les deux task vectors
- Etape 3 (Merge) : sommer les valeurs en gardant uniquement celles dont le signe correspond au signe elu

In [15]:
def ties_merge(tv_a, tv_b, density=0.5):
    # TODO etudiant : implementez TIES merge
    # Etape 1 : Trim - garder uniquement le top density% des valeurs absolues
    # Indice : utilisez torch.kthvalue ou un seuil sur les valeurs absolues
    
    # Etape 2 : Elect Sign - choisir le signe majoritaire par position
    # Indice : le signe majoritaire est celui dont la somme des valeurs absolues est la plus grande
    
    # Etape 3 : Merge - combiner en gardant le signe elect
    # Indice : ne garder que les valeurs dont le signe correspond au signe elu
    pass  # TODO etudiant

print("Exercice a completer : implementez TIES merge")
print("Parametres a tester : density=0.3, density=0.5, density=0.7")

Exercice a completer : testez LERP/SLERP avec differents alpha
Valeurs a tester : [0.2, 0.5, 0.8]
Etapes :
  1) Rechargez le modele de base et entrainez les 2 adaptateurs
  2) Pour chaque alpha, creez un modele LERP et SLERP
  3) Testez sur des questions tech ET cuisine
  4) Observez comment la balance change avec alpha


### Exercice 3 : Ajouter un troisieme expert au systeme de routing

Creez un troisieme adaptateur specialise dans un domaine de votre choix (sport, musique, histoire, etc.) et ajoutez-le au systeme de routing.

**Indices** :
- Etape 1 : Definissez un dataset SFT de 5 exemples pour votre domaine
- Etape 2 : Entrainez le 3e adaptateur LoRA avec le meme lora_config
- Etape 3 : Ajoutez des exemples d'entrainement au routeur (label=2)
- Etape 4 : Reentrainez le routeur avec 3 classes
- Etape 5 : Testez le routing sur des questions des 3 domaines

In [16]:
# Exercice 3 : creez un 3e adapter et mettez a jour le router
# TODO etudiant : definissez votre domaine et vos exemples SFT

# Etape 1 : Definir le dataset SFT pour le 3e domaine
mon_domaine = "sport"  # TODO etudiant : choisissez votre domaine
mes_exemples = [
    # TODO etudiant : ajoutez 5 exemples QA pour votre domaine
]

# Etape 2 : Entrainer le 3e adapter
# TODO etudiant : utilisez le meme pattern que les cellules 5-6

# Etape 3 : Ajouter au router
# TODO etudiant : ajoutez des exemples label=2 dans router_train_data

# Etape 4 : Reentrainer le router avec 3 classes
# TODO etudiant : reentrainez avec num_experts=3

print("Exercice a completer : ajoutez un 3e expert au systeme de routing")

Exercice a completer : implementez TIES merge
Parametres a tester : density=0.3, density=0.5, density=0.7


## 8. Resume du FT-05

| Concept | Detail |
|---------|--------|
| **Task Vectors** | Difference entre poids fine-tunes et poids de base, capture la competence ajoutee |
| **LERP** | Interpolation lineaire : moyenne ponderee des poids, simple mais peut dilater l'espace |
| **SLERP** | Interpolation spherique : preserve les directions, meilleur merge pour 2-3 modeles |
| **DARE** | Drop And Rescale : supprime aleatoirement des poids pour reduire les interferences |
| **TIES** | Trim + Elect Sign + Merge : resout les conflits de signe entre task vectors |
| **Routing (MoE)** | Routeur qui selectionne dynamiquement l'expert adapte a chaque requete |
| **Trade-off** | Merge = 1 modele (moins cher) vs Routing = N modeles (meilleure qualite) |
| **Outils** | Mergekit (production), implementation manuelle PyTorch (ce notebook) |

---

**Navigation** : [FT-01](./FT-01-Introduction-FineTuning.ipynb) | [FT-02](./FT-02-QLoRA-Quantization.ipynb) | [FT-03](./FT-03-Supervised-FineTuning-SFT.ipynb) | [FT-04](./FT-04-RLHF-DPO.ipynb) | **FT-05**